# Document Similarity — Semantic Search in Practice

## What is Document Similarity?

Document similarity is the ability to find which documents are most relevant to a given query, based on **meaning** rather than keyword matching.

**Keyword search** (old way):
- Query: "furry animal on roof" 
- Finds: only documents containing the exact words "furry", "animal", "roof"
- Misses: "cat sat on the mat" (same meaning, different words)

**Semantic search** (embedding way):
- Query: "furry animal on roof"
- Converts both query and documents to vectors
- Measures how "close" the vectors are
- Finds: "cat sat on the mat" ✓ (similar meaning, similar vector)

## How cosine similarity works

**Cosine similarity** measures the angle between two vectors:
- `1.0` = pointing in exactly the same direction = identical meaning
- `0.0` = at 90° = completely unrelated
- `-1.0` = pointing opposite directions = opposite meaning

```python
from sklearn.metrics.pairwise import cosine_similarity

score = cosine_similarity([query_vector], [doc_vector])[0][0]
# score close to 1.0 = very relevant
# score close to 0.0 = unrelated
```

## What this notebook demonstrates

This notebook embeds a "book" of 100 short story pages, then runs semantic search against them. You'll see that the correct page is found even when the query uses completely different words than the actual text.

## What you'll learn

- How to embed a large collection of documents with `embed_documents()`
- How to embed a query and compare it to all documents
- How to rank documents by similarity score
- Why semantic search beats keyword search

## Prerequisites

- `OPENAI_API_KEY` in `.env`
- Virtual environment activated

In [1]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


/Users/rahuljauhari/Desktop/GenAI - Learning/Langchain/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=32)

In [4]:
documents = [
    "Page 1: The little fox woke up early and peeked out of his den. The forest was covered in morning mist, and the world felt brand new.",
    "Page 2: A blue bird sang a cheerful song from the tallest tree. Its melody drifted across the meadow, waking the sleepy flowers.",
    "Page 3: The river gurgled happily as it carried fallen leaves downstream. A family of ducks paddled quickly to catch up with each other.",
    "Page 4: The old oak tree stood proudly in the center of the forest. It had watched generations of animals grow and play beneath its branches.",
    "Page 5: A curious rabbit hopped through the tall grass. Its ears twitched at every sound, listening carefully for danger.",
    "Page 6: The sun began to rise higher, turning the sky golden. Butterflies stretched their wings and danced in the light.",
    "Page 7: Deep in the cave, glowing crystals lit the walls. A small hedgehog marveled at the shimmering colors.",
    "Page 8: The little squirrel gathered acorns one by one. He stacked them neatly, dreaming of cozy winter nights.",
    "Page 9: In the village, children laughed as they chased each other around the square. Their joy echoed like bells in the air.",
    "Page 10: An old storyteller sat by the fire. His voice carried tales of dragons, heroes, and lands beyond the sea.",
    "Page 11: The rain tapped gently on the rooftops. Each drop felt like a lullaby, soothing the town to sleep.",
    "Page 12: A rainbow arched across the sky after the storm. Everyone stopped to admire its seven shining colors.",
    "Page 13: The farmer’s horse trotted slowly down the dusty road. Its hooves left little clouds behind with every step.",
    "Page 14: The little cat curled up in a basket. She purred softly, dreaming of chasing butterflies.",
    "Page 15: The lanterns glowed warmly in the village square. They lit up the night like tiny stars on Earth.",
    "Page 16: A gentle breeze carried the scent of jasmine. It made the travelers smile as they walked the path.",
    "Page 17: The tall castle shimmered in the distance. Its towers looked like they touched the clouds.",
    "Page 18: A knight polished his armor carefully. He wanted it to shine bright for the festival tomorrow.",
    "Page 19: The baker pulled hot bread from the oven. The smell filled the street and made everyone’s stomach growl.",
    "Page 20: The little mouse peeked from his hole. He sniffed the air, hoping for crumbs from the baker’s shop.",
    "Page 21: The stars twinkled like diamonds. A young dreamer wished upon the brightest one she could see.",
    "Page 22: The sea roared loudly against the rocks. A lighthouse stood firm, guiding sailors home.",
    "Page 23: The dragon stretched its wings. Despite its fierce looks, it only wanted to guard its treasure of books.",
    "Page 24: A gentle deer drank from the stream. Its reflection shimmered like silver in the water.",
    "Page 25: The shepherd’s dog barked happily. He was proud of keeping all the sheep safe.",
    "Page 26: The little girl held her grandmother’s hand. They walked together, sharing stories of long ago.",
    "Page 27: The market was full of colors. Stalls overflowed with fruits, fabrics, and spices.",
    "Page 28: A boy flew his kite high above. The red cloth fluttered like a bird in the open sky.",
    "Page 29: The forest path twisted and turned. Each corner promised a new adventure.",
    "Page 30: The wizard stirred his pot of stew. Sparks of magic bubbled up, making the soup glow.",
    "Page 31: The snow covered everything in white. Children built snowmen and laughed together.",
    "Page 32: The fireplace crackled warmly. A family gathered close, telling stories before bed.",
    "Page 33: The moon lit the garden softly. Fireflies joined in, twinkling like tiny lanterns.",
    "Page 34: The fisherman rowed his boat gently. He hummed a song as he cast his net.",
    "Page 35: The painter dipped her brush in blue. She painted the morning sky on her canvas.",
    "Page 36: The puppy chased its tail round and round. Everyone laughed at its silly dance.",
    "Page 37: The traveler opened his map. The journey ahead looked long, but full of wonder.",
    "Page 38: The baker’s daughter offered a warm pie. The smell of apples filled the air with sweetness.",
    "Page 39: The children played hide and seek. Their giggles gave away their hiding spots.",
    "Page 40: The owl hooted from the treetop. It kept watch as the night grew darker.",
    "Page 41: The stars drew patterns across the sky. Some said they told old legends if you looked closely.",
    "Page 42: The princess picked wildflowers. She braided them into her golden hair.",
    "Page 43: The cobbler worked late in his shop. Each shoe he made carried a piece of his heart.",
    "Page 44: The blacksmith hammered sparks into the air. His strength shaped iron into tools for the village.",
    "Page 45: The fairground filled with laughter. Rides spun, music played, and lights dazzled everyone.",
    "Page 46: The little fox curled into a ball. The cold night air made his fur puff up warmly.",
    "Page 47: The storyteller opened a new book. This tale was about courage, kindness, and friendship.",
    "Page 48: The fisherman caught a golden fish. To his surprise, it began to speak.",
    "Page 49: The children lit paper lanterns. They floated into the night sky like glowing dreams.",
    "Page 50: The castle bells rang loudly. A great celebration was about to begin.",
    "Page 51: The gentle rain kissed the earth. Flowers smiled and drank happily.",
    "Page 52: The magician pulled a rabbit from his hat. The crowd gasped with delight.",
    "Page 53: The sailor gazed at the horizon. He wondered what new lands awaited him.",
    "Page 54: The farmer’s field bloomed with sunflowers. Their golden heads followed the sun.",
    "Page 55: The baker’s cat stretched lazily. She yawned, then returned to her nap.",
    "Page 56: The stars looked brighter on the mountain. Travelers whispered their wishes to them.",
    "Page 57: The puppy found a stick bigger than itself. It proudly carried it home.",
    "Page 58: The market smelled of cinnamon and oranges. Everyone carried baskets full of treats.",
    "Page 59: The children splashed in puddles. Their laughter was as bright as the rainbows above.",
    "Page 60: The old library smelled of parchment. Each book held a secret waiting to be discovered.",
    "Page 61: The toy maker smiled at his creations. Each doll and puppet seemed alive with joy.",
    "Page 62: The horse galloped freely across the meadow. Its mane flew like a banner of freedom.",
    "Page 63: The gentle wind carried a tune. Somewhere, someone was playing a flute.",
    "Page 64: The fisherman’s lantern glowed on the water. Fish shimmered beneath the waves.",
    "Page 65: The mountain echoed with songbirds. Their music welcomed the morning sun.",
    "Page 66: The baker shared cookies with children. Their hands were sticky with sugar, but their smiles were bright.",
    "Page 67: The painter mixed red and gold. She captured the sunset in her picture.",
    "Page 68: The carpenter carved a wooden bird. Its wings looked ready to fly.",
    "Page 69: The forest whispered secrets. The trees seemed to lean closer, listening.",
    "Page 70: The knight bowed politely. Even in armor, he carried kindness in his heart.",
    "Page 71: The black cat sat on the windowsill. Its eyes gleamed like emeralds in the dark.",
    "Page 72: The garden bloomed with roses. Their fragrance lingered in the warm air.",
    "Page 73: The old clock chimed midnight. Somewhere, a new adventure began.",
    "Page 74: The fisherman’s net caught only shells. But he smiled, for they were beautiful treasures.",
    "Page 75: The children built a fort from blankets. It became their castle for the night.",
    "Page 76: The candlelight flickered gently. Shadows danced across the walls.",
    "Page 77: The meadow was full of daisies. Bees hummed happily among them.",
    "Page 78: The castle doors creaked open. Inside, a grand hall sparkled with chandeliers.",
    "Page 79: The wise owl spoke in riddles. The children listened carefully to learn his wisdom.",
    "Page 80: The forest path led to a hidden pond. Frogs croaked as fireflies lit the surface.",
    "Page 81: The sailor’s compass pointed north. His journey had only just begun.",
    "Page 82: The windmill turned slowly in the breeze. Its blades creaked like an old song.",
    "Page 83: The puppy discovered a butterfly. He chased it, but it always flew just out of reach.",
    "Page 84: The baker decorated a cake with care. It was a gift for the village festival.",
    "Page 85: The little bird built a nest. It carefully lined it with twigs and feathers.",
    "Page 86: The storyteller ended the tale with a smile. Everyone clapped with joy.",
    "Page 87: The castle tower overlooked the valley. The view stretched far into the distance.",
    "Page 88: The rainbow faded slowly. But the memory of its colors stayed in their hearts.",
    "Page 89: The children planted seeds in the soil. They dreamed of flowers growing tall.",
    "Page 90: The gentle stream sang softly. Pebbles glistened beneath its clear water.",
    "Page 91: The blacksmith’s forge glowed red. Sparks danced like fireflies in the dark.",
    "Page 92: The kitten played with a ball of yarn. It rolled and tumbled, making a happy mess.",
    "Page 93: The castle feast was grand. Tables overflowed with fruits, pies, and laughter.",
    "Page 94: The storyteller closed the book. But the magic of the story lingered on.",
    "Page 95: The children drifted to sleep. They dreamed of dragons, heroes, and faraway lands.",
    "Page 96: The moon shone gently above. Its silver light wrapped the world in peace.",
    "Page 97: The dawn arrived with golden light. Birds greeted the day with joyful song.",
    "Page 98: The market packed up slowly. Stalls closed as lanterns lit the quiet streets.",
    "Page 99: The little fox returned to his den. He curled up, tired but happy.",
    "Page 100: And so the storybook ended for now. But tomorrow promised new adventures."
]


In [5]:
embeddings = embedding_model.embed_documents(documents)

In [6]:
query = "Where did the kitten go?"
query_embedding = embedding_model.embed_query(query)

In [9]:
scores = cosine_similarity([query_embedding], embeddings)

In [10]:
scores[0]

array([ 2.18610814e-01,  1.03540862e-01,  1.87940428e-01,  2.96893006e-01,
        3.83270622e-01,  5.84952176e-02,  4.46373816e-01,  2.76696955e-01,
        4.19771766e-02,  1.69049039e-01,  2.87373047e-02,  1.95505976e-01,
        8.68503656e-02,  4.84318092e-01, -2.09339265e-02, -3.23520109e-02,
        3.36627602e-01,  1.27441473e-01,  2.45551678e-01,  3.43504627e-01,
        2.75755472e-01,  9.75417690e-02,  4.09820202e-01,  3.13104239e-01,
        3.55931908e-01,  3.38914020e-02, -2.60830169e-01,  2.41795810e-01,
       -1.34922993e-01,  4.72683254e-02, -5.07727756e-02, -1.72179603e-02,
        2.28599903e-01,  1.98221306e-01,  1.03446739e-01,  3.23234013e-01,
        1.66668527e-01,  7.50013906e-02,  1.63570129e-01,  4.17353852e-01,
        1.69037430e-01,  1.78682771e-01,  2.73765970e-01,  2.98817296e-01,
       -1.86078571e-02,  4.81745955e-01,  2.45452201e-01,  2.54772309e-01,
        1.34429158e-01, -5.16810970e-02,  1.75238442e-02,  2.05037394e-01,
        3.14828748e-01, -

In [11]:
scores[0][0]

np.float64(0.218610814415233)

In [ ]:
index, score = sorted(list(enumerate(scores[0])), key=lambda x: x[1])[-1]

In [21]:
print(documents[index])
print("Similariity Score:", score)
print("Query:", query)

Page 92: The kitten played with a ball of yarn. It rolled and tumbled, making a happy mess.
Similariity Score: 0.5735901458207477
Query: Where did the kitten go?
